In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/Users/naren/Projects/attrition-payroll-risk/data/raw/WA_Fn-UseC_-HR-Employee-Attrition.xls', encoding='utf-8-sig')

# Convert target to binary
df['Attrition_Flag'] = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)

print("Shape:", df.shape)
print("Attrition Flag:\n", df['Attrition_Flag'].value_counts())

Shape: (1470, 36)
Attrition Flag:
 Attrition_Flag
0    1233
1     237
Name: count, dtype: int64


In [2]:
# These columns have same value for everyone - no predictive power
df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'], inplace=True)

print("Shape after dropping:", df.shape)

Shape after dropping: (1470, 33)


Payroll Feature 1: Compensation Ratio

In [3]:
# Average income per job level
avg_income_by_level = df.groupby('JobLevel')['MonthlyIncome'].transform('mean')

# How much does this employee earn vs their grade average?
df['Compensation_Ratio'] = df['MonthlyIncome'] / avg_income_by_level

print("Compensation Ratio Stats:")
print(df.groupby('Attrition')['Compensation_Ratio'].mean())

Compensation Ratio Stats:
Attrition
No     1.007574
Yes    0.960594
Name: Compensation_Ratio, dtype: float64


Payroll Feature 2: Salary Hike Band

In [4]:
def hike_band(hike):
    if hike <= 11:
        return 'Low'
    elif hike <= 14:
        return 'Medium'
    else:
        return 'High'

df['Hike_Band'] = df['PercentSalaryHike'].apply(hike_band)

print(pd.crosstab(df['Hike_Band'], df['Attrition'], normalize='index') * 100)

Attrition         No        Yes
Hike_Band                      
High       83.895706  16.104294
Low        80.476190  19.523810
Medium     85.032895  14.967105


Payroll Feature 2: Salary Hike Band

In [5]:
def hike_band(hike):
    if hike <= 11:
        return 'Low'
    elif hike <= 14:
        return 'Medium'
    else:
        return 'High'

df['Hike_Band'] = df['PercentSalaryHike'].apply(hike_band)

print(pd.crosstab(df['Hike_Band'], df['Attrition'], normalize='index') * 100)

Attrition         No        Yes
Hike_Band                      
High       83.895706  16.104294
Low        80.476190  19.523810
Medium     85.032895  14.967105


In [6]:
#Payroll Feature 3: Overtime Risk Score

In [7]:
df['OverTime_Flag'] = df['OverTime'].apply(lambda x: 1 if x == 'Yes' else 0)

# Combined risk: overtime AND low hike
df['Overtime_LowHike_Risk'] = (
    (df['OverTime_Flag'] == 1) & (df['Hike_Band'] == 'Low')
).astype(int)

print("Overtime + Low Hike Risk vs Attrition:")
print(pd.crosstab(df['Overtime_LowHike_Risk'], df['Attrition'], normalize='index') * 100)

Overtime + Low Hike Risk vs Attrition:
Attrition                     No        Yes
Overtime_LowHike_Risk                      
0                      84.740951  15.259049
1                      63.934426  36.065574


In [8]:
#Payroll Feature 4: Tenure-to-Promotion Gap

In [9]:
# How many years per job level - higher means stagnation
df['Tenure_Per_Level'] = df['YearsAtCompany'] / (df['JobLevel'] + 1)

print("Tenure Per Level vs Attrition:")
print(df.groupby('Attrition')['Tenure_Per_Level'].mean())

Tenure Per Level vs Attrition:
Attrition
No     2.291768
Yes    1.786498
Name: Tenure_Per_Level, dtype: float64


In [10]:
#Payroll Feature 5: Total Compensation Score

In [11]:
# Normalize and combine compensation signals
df['Total_Comp_Score'] = (
    df['MonthlyIncome'] * 0.6 +
    df['StockOptionLevel'] * 1000 +
    df['PercentSalaryHike'] * 100
)

print("Total Comp Score vs Attrition:")
print(df.groupby('Attrition')['Total_Comp_Score'].mean())

Total Comp Score vs Attrition:
Attrition
No     6467.851419
Yes    4909.386498
Name: Total_Comp_Score, dtype: float64


In [12]:
#Payroll Feature 6: Experience-to-Pay Ratio

In [13]:
# Avoid division by zero
df['Exp_Pay_Ratio'] = df['MonthlyIncome'] / (df['TotalWorkingYears'] + 1)

print("Experience to Pay Ratio vs Attrition:")
print(df.groupby('Attrition')['Exp_Pay_Ratio'].mean())

Experience to Pay Ratio vs Attrition:
Attrition
No     575.382922
Yes    651.007062
Name: Exp_Pay_Ratio, dtype: float64


In [14]:
#Encode Categorical Columns

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

cat_cols = ['BusinessTravel', 'Department', 'EducationField',
            'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'Hike_Band']

for col in cat_cols:
    df[col + '_Enc'] = le.fit_transform(df[col])

print("Encoding done!")
print(df.shape)

Encoding done!
(1470, 48)


In [16]:
df.to_csv('/Users/naren/Projects/attrition-payroll-risk/data/processed/attrition_processed.csv', index=False)
print("Processed data saved!")
print("Final Shape:", df.shape)
print("\nNew Features Added:")
new_features = ['Attrition_Flag', 'Compensation_Ratio', 'Hike_Band',
                'OverTime_Flag', 'Overtime_LowHike_Risk',
                'Tenure_Per_Level', 'Total_Comp_Score', 'Exp_Pay_Ratio']
print(new_features)

Processed data saved!
Final Shape: (1470, 48)

New Features Added:
['Attrition_Flag', 'Compensation_Ratio', 'Hike_Band', 'OverTime_Flag', 'Overtime_LowHike_Risk', 'Tenure_Per_Level', 'Total_Comp_Score', 'Exp_Pay_Ratio']
